# Etapa 4: Modelado — Predicción de Churn

Este notebook corresponde a la Etapa 4 del ciclo CRISP-DM del proyecto. Parte del dataset
ya limpio y transformado en la Etapa 3 (`data/processed/X_features.csv` y `y_target.csv`)
para entrenar y comparar tres algoritmos de clasificación: Regresión Logística, Random
Forest y Gradient Boosting.

El objetivo no es maximizar una métrica en abstracto, sino identificar qué modelo detecta
mejor a los clientes en riesgo real de fuga (Recall, Precision, F1), evaluado con
Stratified K-Fold Cross-Validation para obtener una estimación robusta del desempeño.
El modelo (o modelos) seleccionados acá van a ser la base de la Etapa 5, donde se traduce
su desempeño en impacto económico real para el negocio.

**Nota**: el ajuste del umbral de decisión (más allá del 0.5 por default) se deja para la
Etapa 5, una vez definida la lógica de costos y beneficios.

In [1]:
import pandas as pd

X = pd.read_csv('../data/processed/X_features.csv')
y = pd.read_csv('../data/processed/y_target.csv')

In [2]:
print(X.shape)
print(y.shape)

(7043, 29)
(7043, 1)


## Train/Test Split

Se separa el dataset completo en dos partes: un 80% para desarrollo (entrenamiento y
cross-validation de los distintos modelos) y un 20% que queda completamente aislado
como "examen final", sin participar en ninguna decisión durante el desarrollo del modelo.

La división usa `stratify=y` para preservar la proporción real de churn (73.46%/26.54%)
tanto en train como en test, evitando que la partición al azar genere un desbalance
distinto entre ambos grupos. `random_state=42` fija la aleatoriedad para que la división
sea siempre la misma cada vez que se corra el notebook, garantizando reproducibilidad.

In [3]:
from sklearn.model_selection import train_test_split

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, stratify=y, random_state=42)

In [4]:
print(X_train.shape)
print(X_test.shape)
print(y_train['Churn Value'].mean())
print(y_test['Churn Value'].mean())

(5634, 29)
(1409, 29)
0.2653532126375577
0.2654364797728886


### Resultado: Train/Test Split

Se confirma la división 80/20 (5,634 filas train / 1,409 filas test), con la proporción
de churn prácticamente idéntica en ambos grupos (26.54% train, 26.54% test) — el
parámetro `stratify=y` cumplió su función correctamente. El set de test queda apartado
y no se vuelve a tocar hasta la evaluación final del modelo elegido.

## Configurar Stratified K-Fold

Se configura el objeto Stratified K-Fold que se va a usar para evaluar cada modelo
durante el desarrollo, dividiendo el set de entrenamiento (X_train, y_train) en 5
partes que mantienen la proporción real de churn (73%/27%) en cada una. A diferencia
del Train/Test Split del paso anterior, acá no se dividen los datos todavía — se define
la configuración que se va a reutilizar repetidamente en el paso 4, al entrenar y
comparar los tres modelos.

In [5]:
from sklearn.model_selection import StratifiedKFold

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

### Resultado: Stratified K-Fold configurado

Se configuró `skf` con 5 folds, mezcla aleatoria y semilla fija (random_state=42).
Este objeto no divide los datos por sí mismo — define la estrategia que se va a
aplicar en el paso 4 al entrenar y comparar los tres modelos.

## Entrenar y comparar los 3 modelos

Se entrenan Regresión Logística, Random Forest y Gradient Boosting usando el mismo
Stratified K-Fold (5 folds), registrando las mismas métricas para los tres en igualdad
de condiciones (Recall, Precision, F1, ROC-AUC de la clase Churn=1). Como es el mismo
proceso repetido para cada modelo, se escribe una única función reutilizable en
`src/modeling.py`, siguiendo el principio DRY (Don't Repeat Yourself).

### Primero cross_validate con un solo modelo (Regresión Logística)

Antes de armar la función reutilizable para los 3 modelos, se prueba `cross_validate`
con un solo modelo (Regresión Logística) para entender su funcionamiento: en cada una
de las 5 rondas del Stratified K-Fold, entrena un modelo nuevo con 4 folds y lo evalúa
con el fold restante, registrando Recall, Precision, F1 y ROC-AUC de la clase Churn=1
en cada ronda.

La primera corrida arrojó un `ConvergenceWarning` repetido en las 5 rondas: el modelo
no logró estabilizarse dentro de las 1000 iteraciones permitidas. La causa es que las
variables numéricas del dataset están en escalas muy distintas entre sí (columnas
binarias de 0-1 conviven con `Total Charges`, que llega a ~8,000), algo ya anticipado
en el planning de la Etapa 3.

La corrección usa un `Pipeline` que encadena `StandardScaler` y el modelo como un solo
objeto — así, el escalado se recalcula dentro de cada fold usando solo sus datos de
entrenamiento, evitando que información del fold de evaluación se filtre en el cálculo
del escalado (data leakage).

In [6]:
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import cross_validate
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline

modelo_escalado = Pipeline([
    ('scaler', StandardScaler()),
    ('modelo', LogisticRegression(class_weight='balanced', random_state=42, max_iter=1000))
])

metricas = ['recall', 'precision', 'f1', 'roc_auc']

resultados = cross_validate(
    modelo_escalado, 
    X_train, 
    y_train['Churn Value'], 
    cv=skf, 
    scoring=metricas
)

resultados

{'fit_time': array([0.02596641, 0.02891088, 0.02443194, 0.0305326 , 0.02667904]),
 'score_time': array([0.01260448, 0.01282048, 0.01239538, 0.0129838 , 0.0123992 ]),
 'test_recall': array([0.83946488, 0.77257525, 0.78929766, 0.85618729, 0.7993311 ]),
 'test_precision': array([0.5504386 , 0.51910112, 0.50752688, 0.53222453, 0.5456621 ]),
 'test_f1': array([0.66490066, 0.62096774, 0.61780105, 0.65641026, 0.64857531]),
 'test_roc_auc': array([0.86212092, 0.837932  , 0.85335175, 0.87106983, 0.87069757])}

#### Promedio de las métricas de Regresión Logística

Se calcula el promedio de las 5 rondas para cada métrica, resumiendo el desempeño
general del modelo en un solo número por métrica (en vez de tener que leer los 5
valores sueltos cada vez).

In [7]:
for metrica in metricas:
    promedio = resultados[f'test_{metrica}'].mean()
    print(f"{metrica}: {promedio:.4f}")

recall: 0.8114
precision: 0.5310
f1: 0.6417
roc_auc: 0.8590


##### Resultado: Regresión Logística (con class_weight='balanced')

Recall: 0.8114 | Precision: 0.5310 | F1: 0.6417 | ROC-AUC: 0.8590

El modelo detecta correctamente el 81% de los clientes que realmente se van a ir, a
costa de una Precision más baja (53%) — consistente con `class_weight='balanced'`,
que prioriza detectar la clase minoritaria aunque genere más falsos positivos. El
ROC-AUC de 0.86 indica una buena capacidad general de separación entre ambas clases.

### Random Forest

Se evalúa Random Forest con el mismo Stratified K-Fold y las mismas métricas que
Regresión Logística, reutilizando la función `evaluar_modelo()` de `src/modeling.py`.
A diferencia de Regresión Logística, no necesita escalado de variables (no usa Pipeline),
ya que los modelos basados en árboles no son sensibles a la escala de los datos.

In [8]:
from sklearn.ensemble import RandomForestClassifier

modelo_rf = RandomForestClassifier(class_weight='balanced', random_state=42)

In [9]:
import sys
sys.path.append('..')

from src.modeling import evaluar_modelo

resultados_rf = evaluar_modelo(modelo_rf, X_train, y_train['Churn Value'], skf, metricas)

Resultados promedio:
recall: 0.6689
precision: 0.5857
f1: 0.6245
roc_auc: 0.8430


#### Resultado: Random Forest (con class_weight='balanced')

Recall: 0.6689 | Precision: 0.5857 | F1: 0.6245 | ROC-AUC: 0.8430

Comparado con Regresión Logística, Random Forest muestra menor Recall (67% vs 81%) pero
mayor Precision (59% vs 53%) — detecta menos casos reales de churn, pero se equivoca
menos cuando marca a alguien como riesgo. El F1 y ROC-AUC quedan parejos entre ambos
modelos, sin un ganador claro todavía.

### Gradient Boosting

Se evalúa Gradient Boosting con el mismo Stratified K-Fold y las mismas métricas que
los dos modelos anteriores, reutilizando `evaluar_modelo()`. Al igual que Random Forest,
no necesita escalado de variables por ser un modelo basado en árboles.

In [10]:
from sklearn.ensemble import GradientBoostingClassifier

modelo_gb = GradientBoostingClassifier(random_state=42)

In [11]:
resultados_gb = evaluar_modelo(modelo_gb, X_train, y_train['Churn Value'], skf, metricas)

Resultados promedio:
recall: 0.5472
precision: 0.6593
f1: 0.5977
roc_auc: 0.8625


#### Resultado: Gradient Boosting (sin ajuste de balance)

Recall: 0.5472 | Precision: 0.6593 | F1: 0.5977 | ROC-AUC: 0.8625

Es el modelo más conservador de los tres: menor Recall pero mayor Precision. Sin
embargo, tiene el ROC-AUC más alto (86.25%), indicando la mejor capacidad de separación
general entre clases — una señal de que el ajuste del umbral de decisión en la Etapa 5
podría mejorar significativamente su Recall sin perder esa buena separación de base.

## Comparación de los 3 modelos

Se consolidan los resultados promedio de Regresión Logística, Random Forest y Gradient
Boosting en una sola tabla, para facilitar la comparación directa antes de decidir cuál
(o cuáles) avanzan a la Etapa 5.

In [12]:
comparacion_resultados = pd.DataFrame({
    'Logistic Regression': [resultados[f'test_{metrica}'].mean() for metrica in metricas],
    'Random Forest': [resultados_rf[f'test_{metrica}'].mean() for metrica in metricas],
    'Gradient Boosting': [resultados_gb[f'test_{metrica}'].mean() for metrica in metricas]
}, index=metricas)

comparacion_resultados

,Logistic Regression,Random Forest,Gradient Boosting
recall,0.811371,0.668896,0.547157
precision,0.530991,0.585675,0.659305
f1,0.641731,0.624479,0.597667
roc_auc,0.859034,0.842971,0.862539


### Resultado: tabla comparativa

|              | Regresión Logística | Random Forest | Gradient Boosting |
|--------------|---------------------|----------------|---------------------|
| Recall       | 0.8114              | 0.6689         | 0.5472               |
| Precision    | 0.5310              | 0.5857         | 0.6593               |
| F1           | 0.6417              | 0.6245         | 0.5977               |
| ROC-AUC      | 0.8590              | 0.8430         | 0.8625               |

No hay un ganador absoluto: Regresión Logística domina en Recall, Gradient Boosting en
Precision y ROC-AUC, y los tres quedan relativamente parejos en F1. La decisión de qué
modelo(s) avanzan a la Etapa 5 depende del contexto de negocio (impacto económico de
Falsos Positivos vs. Falsos Negativos), no solo de estas métricas en abstracto.

## Gráfico: comparación visual de los 3 modelos

Se grafica la tabla comparativa como barras agrupadas, para visualizar de forma más
directa los trade-offs entre modelos (Recall vs. Precision, especialmente).

In [13]:
comparacion_larga = comparacion_resultados.reset_index().melt(id_vars='index', var_name='Modelo', value_name='Valor')
comparacion_larga.columns = ['Métrica', 'Modelo', 'Valor']
comparacion_larga

,Métrica,Modelo,Valor
0,recall,Logistic Regression,0.811371
1,precision,Logistic Regression,0.530991
2,f1,Logistic Regression,0.641731
3,roc_auc,Logistic Regression,0.859034
4,recall,Random Forest,0.668896
5,precision,Random Forest,0.585675
6,f1,Random Forest,0.624479
7,roc_auc,Random Forest,0.842971
8,recall,Gradient Boosting,0.547157
9,precision,Gradient Boosting,0.659305


### Resultado: transformación a formato largo

Se convirtió la tabla comparativa de formato ancho (4 filas × 3 columnas) a formato
largo (12 filas × 3 columnas) usando `.melt()`, donde cada fila representa una
combinación única de métrica + modelo + valor — el formato que requiere Plotly para
graficar barras agrupadas.

In [14]:
from plotly import express as px

fig = px.bar(
    comparacion_larga,
    x='Métrica',
    y='Valor',
    color='Modelo',
    barmode='group',
    title='Comparación de Modelos por Métrica',
    text_auto='.2f'
)
fig.show()

## Selección de modelos finalistas

Se seleccionan dos modelos para continuar a la Etapa 4.7 (feature importance) y a la
Etapa 5 (impacto económico): **Regresión Logística** (mejor Recall, 81%) y **Gradient
Boosting** (mejor ROC-AUC y Precision, con potencial de mejora en Recall al ajustar el
umbral de decisión). Random Forest queda descartado por no destacarse en ninguna métrica
frente a los otros dos.

La decisión final entre los dos finalistas se toma en la Etapa 5, comparando el ahorro
neto económico de cada uno con su umbral óptimo — un criterio de negocio real, en vez
de depender solo de métricas técnicas en abstracto.

## Paso 4.7: Feature Importance de Gradient Boosting

Se entrena Gradient Boosting una vez sobre todo X_train (no en folds, como en
cross-validation) para poder extraer e inspeccionar `feature_importances_` — el
puntaje que el modelo le asigna a cada variable según cuánto la usó para tomar
decisiones. Esto permite confirmar si las variables que ya identificamos como fuertes
en Excel y en los gráficos (Contract, tenure, Payment Method) también son las más
relevantes según el modelo.

In [15]:
modelo_gb.fit(X_train, y_train['Churn Value'])

importancias = pd.DataFrame({
    'Característica': X_train.columns,
    'Importancia': modelo_gb.feature_importances_
}).sort_values(by='Importancia', ascending=False)

importancias

,Característica,Importancia
12,Contract,0.387123
4,Tenure Months,0.142565
20,Internet Service_Fiber optic,0.102761
3,Dependents,0.092289
24,Payment Method_Electronic check,0.050669
15,Total Charges,0.048618
14,Monthly Charges,0.035624
27,Cargo_Promedio_Mensual,0.032821
21,Internet Service_No,0.027559
13,Paperless Billing,0.016229


## Paso 4.7 (continuación): Coeficientes de Regresión Logística

A diferencia de `feature_importances_` (que solo da magnitud), los coeficientes de
Regresión Logística indican también la **dirección** del efecto: un coeficiente
positivo significa que a mayor valor de esa variable, mayor probabilidad de churn;
uno negativo, el efecto contrario. Esto permite entender no solo qué variables pesan,
sino cómo empujan la predicción en cada sentido.

In [16]:
modelo_escalado.fit(X_train, y_train['Churn Value'])

coeficientes = pd.DataFrame({
    'Variable': X_train.columns,
    'Coeficiente': modelo_escalado.named_steps['modelo'].coef_[0]
}).sort_values('Coeficiente', ascending=False)

coeficientes

,Variable,Coeficiente
28,Grupo_Antiguedad,0.702877
20,Internet Service_Fiber optic,0.580710
15,Total Charges,0.455607
11,Streaming Movies,0.215214
10,Streaming TV,0.208645
13,Paperless Billing,0.158520
2,Partner,0.142988
27,Cargo_Promedio_Mensual,0.130785
24,Payment Method_Electronic check,0.127701
18,Multiple Lines_Yes,0.100673


### Corrección: eliminar Grupo_Antiguedad por multicolinealidad

Se elimina `Grupo_Antiguedad` de X_train y X_test, ya que es una versión resumida de
`Tenure Months` (creada a partir de la misma columna con `pd.cut()` en la Etapa 3).
Mantener ambas generaba multicolinealidad en Regresión Logística, evidenciada por
coeficientes contradictorios entre sí. Se conserva `Tenure Months` por ser la versión
con mayor precisión (valor exacto vs. rango agrupado).

In [17]:
X_train = X_train.drop(columns=['Grupo_Antiguedad'])
X_test = X_test.drop(columns=['Grupo_Antiguedad'])

print(X_train.shape)

(5634, 28)


In [18]:
print(X_test.shape)

(1409, 28)


### Re-evaluación con el dataset corregido

Se vuelve a correr `cross_validate` para los dos modelos finalistas usando el
X_train sin `Grupo_Antiguedad`, empezando por Gradient Boosting (mismo orden que
la primera evaluación), para confirmar que las métricas no cambian sustancialmente
sin esa columna redundante.

In [19]:
resultados = evaluar_modelo(modelo_gb, X_train, y_train['Churn Value'], skf, metricas)

Resultados promedio:
recall: 0.5472
precision: 0.6593
f1: 0.5977
roc_auc: 0.8625


#### Resultado: Gradient Boosting sin `Grupo_Antiguedad`

Recall: 0.5472 | Precision: 0.6593 | F1: 0.5977 | ROC-AUC: 0.8625

Idénticos a los valores obtenidos con el dataset original — confirma que
`Grupo_Antiguedad` no aportaba información predictiva adicional a Tenure Months,
tal como se esperaba de dos variables redundantes entre sí.

In [20]:
resultados = evaluar_modelo(modelo_escalado, X_train, y_train['Churn Value'], skf, metricas)

Resultados promedio:
recall: 0.8127
precision: 0.5294
f1: 0.6409
roc_auc: 0.8587


#### Resultado: Regresión Logística sin `Grupo_Antiguedad`

Recall: 0.8127 | Precision: 0.5294 | F1: 0.6409 | ROC-AUC: 0.8587

Prácticamente idénticos a los valores originales (diferencias mínimas de redondeo),
confirmando que la columna redundante no aportaba capacidad predictiva. El cambio
relevante se espera ver en los coeficientes, no en estas métricas.

In [21]:
modelo_escalado.fit(X_train, y_train['Churn Value'])

coeficientes = pd.DataFrame({
    'Variable': X_train.columns,
    'Coeficiente': modelo_escalado.named_steps['modelo'].coef_[0]
}).sort_values('Coeficiente', ascending=False)

coeficientes

,Variable,Coeficiente
20,Internet Service_Fiber optic,0.585130
15,Total Charges,0.505495
11,Streaming Movies,0.219071
10,Streaming TV,0.209185
13,Paperless Billing,0.157478
2,Partner,0.138667
24,Payment Method_Electronic check,0.127811
18,Multiple Lines_Yes,0.099390
27,Cargo_Promedio_Mensual,0.069443
26,Total Services,0.052934


#### Resultado: coeficientes corregidos, sin contradicción

Al eliminar `Grupo_Antiguedad`, `Tenure Months` queda como coeficiente único
(-1.204), sin la contradicción de signos que existía cuando ambas variables
convivían en el modelo. El resto de la tabla se mantiene coherente: Contract,
Internet Service_No y Dependents con efecto negativo sobre el churn (reducen el
riesgo), Internet Service_Fiber optic y Total Charges con efecto positivo
(aumentan el riesgo) — consistente con los hallazgos ya confirmados en Excel y
en los gráficos del notebook 1.

## Paso 8: Evaluación final sobre el set de test

Se entrena cada modelo finalista una vez sobre todo X_train, y se evalúa sobre X_test
(el 20% reservado desde el paso 2, nunca usado en desarrollo ni cross-validation). Esta
es la estimación más realista de cómo se comportaría el modelo con clientes nuevos,
libre de cualquier ajuste hecho durante el desarrollo.

### Evaluación de Gradient Boosting sobre el set de test

Se reentrena Gradient Boosting sobre el X_train completo (28 columnas, ya sin
Grupo_Antiguedad) y se genera una predicción sobre X_test — datos que el modelo
nunca vio durante todo el desarrollo. Esta es la primera de las dos evaluaciones
finales, antes de compararla con Regresión Logística.

In [22]:
modelo_gb.fit(X_train, y_train['Churn Value'])

,"random_state random_state: int, RandomState instance or None, default=NoneControls the random seed given to each Tree estimator at eachboosting iteration.In addition, it controls the random permutation of the features ateach split (see Notes for more details).It also controls the random splitting of the training data to obtain avalidation set if `n_iter_no_change` is not None.Pass an int for reproducible output across multiple function calls.See :term:`Glossary <random_state>`.",42
,"loss loss: {'log_loss', 'exponential'}, default='log_loss'The loss function to be optimized. 'log_loss' refers to binomial andmultinomial deviance, the same as used in logistic regression.It is a good choice for classification with probabilistic outputs.For loss 'exponential', gradient boosting recovers the AdaBoost algorithm.",'log_loss'
,"learning_rate learning_rate: float, default=0.1Learning rate shrinks the contribution of each tree by `learning_rate`.There is a trade-off between learning_rate and n_estimators.Values must be in the range `[0.0, inf)`.For an example of the effects of this parameter and its interaction with``subsample``, see:ref:`sphx_glr_auto_examples_ensemble_plot_gradient_boosting_regularization.py`.",0.1
,"n_estimators n_estimators: int, default=100The number of boosting stages to perform. Gradient boostingis fairly robust to over-fitting so a large number usuallyresults in better performance.Values must be in the range `[1, inf)`.",100
,"subsample subsample: float, default=1.0The fraction of samples to be used for fitting the individual baselearners. If smaller than 1.0 this results in Stochastic GradientBoosting. `subsample` interacts with the parameter `n_estimators`.Choosing `subsample < 1.0` leads to a reduction of varianceand an increase in bias.Values must be in the range `(0.0, 1.0]`.",1.0
,"criterion criterion: {'friedman_mse', 'squared_error'}, default='friedman_mse'This parameter has no effect... versionadded:: 0.18.. deprecated:: 1.9 `criterion` is deprecated and will be removed in 1.11.",'deprecated'
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, values must be in the range `[2, inf)`.- If float, values must be in the range `(0.0, 1.0]` and `min_samples_split` will be `ceil(min_samples_split * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, values must be in the range `[1, inf)`.- If float, values must be in the range `(0.0, 1.0)` and `min_samples_leaf` will be `ceil(min_samples_leaf * n_samples)`... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.Values must be in the range `[0.0, 0.5]`.",0.0
,"max_depth max_depth: int or None, default=3Maximum depth of the individual regression estimators. The maximumdepth limits the number of nodes in the tree. Tune this parameterfor best performance; the best value depends on the interactionof the input variables. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.If int, values must be in the range `[1, inf)`.",3
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease of the impuritygreater than or equal to this value.Values must be in the range `[0.0, inf)`.The weighted impurity decrease equation is the following:: N_t / N * (im

In [23]:
predicciones_gb = modelo_gb.predict(X_test)

#### Métricas finales de Gradient Boosting en test

Se calculan Recall, Precision, F1 y ROC-AUC sobre las predicciones de X_test,
comparando contra los valores reales de y_test. A diferencia del cross-validation,
esta es una sola medición (no un promedio de 5 rondas), ya que el set de test se
usa una única vez.

In [24]:
from sklearn.metrics import recall_score, precision_score, f1_score, roc_auc_score, confusion_matrix

recall_gb = recall_score(y_test, predicciones_gb)
precision_gb = precision_score(y_test, predicciones_gb)
f1_gb = f1_score(y_test, predicciones_gb)

print(f"Recall: {recall_gb:.4f}")
print(f"Precision: {precision_gb:.4f}")
print(f"F1: {f1_gb:.4f}")

Recall: 0.5241
Precision: 0.6469
F1: 0.5790


#### Resultado: métricas finales de Gradient Boosting

Recall: 0.5241 | Precision: 0.6469 | F1: 0.5790

Consistente con los valores de cross-validation (Recall 54.7%, Precision 65.9%, F1
59.8%) — la diferencia de 1-2 puntos porcentuales es esperable y confirma que el
modelo generaliza bien a datos nunca vistos, sin señales de overfitting.

#### ROC-AUC de Gradient Boosting en test

Se calcula ROC-AUC usando las probabilidades predichas (no la clasificación binaria
0/1), ya que esta métrica evalúa la capacidad de separación del modelo a través de
todos los umbrales posibles, no solo en el punto de corte de 0.5.

In [25]:
probabilidades_gb = modelo_gb.predict_proba(X_test)[:, 1]
roc_auc_gb = roc_auc_score(y_test, probabilidades_gb)

print(f"ROC-AUC: {roc_auc_gb:.4f}")

ROC-AUC: 0.8527


#### Resultado: ROC-AUC de Gradient Boosting

ROC-AUC: 0.8527

Consistente con el valor de cross-validation (0.8625) — confirma la buena capacidad
de separación general del modelo entre clientes que se van y que se quedan, incluso
en datos completamente nuevos.

#### Matriz de confusión de Gradient Boosting

Se calcula la matriz de confusión sobre las predicciones de test — los 4 números
reales (Verdaderos Positivos, Falsos Positivos, Verdaderos Negativos, Falsos
Negativos) que en la Etapa 5 se van a traducir directamente en impacto económico,
aplicando los costos y beneficios definidos en el planning original.

In [26]:
matriz_gb = confusion_matrix(y_test, predicciones_gb)
matriz_gb

array([[928, 107],
       [178, 196]])

#### Resultado: matriz de confusión de Gradient Boosting

|                     | Predijo: No se va | Predijo: Se va |
|---------------------|--------------------|-----------------|
| Realidad: No se fue | VN = 928           | FP = 107        |
| Realidad: Se fue    | FN = 178           | VP = 196        |

De los 374 clientes que realmente se fueron en el set de test, el modelo detectó 196
(52.4%) y no detectó 178. De los 1,035 que no se fueron, marcó incorrectamente a 107
como riesgo. Estos 4 valores son la base directa para el cálculo de impacto económico
de la Etapa 5.

### Evaluación de Regresión Logística sobre el set de test

Se reentrena Regresión Logística (dentro de su Pipeline con StandardScaler) sobre el
X_train completo y se genera una predicción sobre X_test — datos nunca vistos durante
el desarrollo. Se calculan las mismas métricas y matriz de confusión que para Gradient
Boosting, para comparar ambos modelos en igualdad de condiciones antes de la decisión
final en la Etapa 5.

In [27]:
modelo_escalado.fit(X_train, y_train['Churn Value'])
predicciones_lr = modelo_escalado.predict(X_test)
probabilidades_lr = modelo_escalado.predict_proba(X_test)[:, 1]

In [28]:
recall_lr = recall_score(y_test, predicciones_lr)
precision_lr = precision_score(y_test, predicciones_lr)
f1_lr = f1_score(y_test, predicciones_lr)
roc_auc_lr = roc_auc_score(y_test, probabilidades_lr)

print(f"Recall: {recall_lr:.4f}")
print(f"Precision: {precision_lr:.4f}")
print(f"F1: {f1_lr:.4f}")
print(f"ROC-AUC: {roc_auc_lr:.4f}")

Recall: 0.7834
Precision: 0.5105
F1: 0.6181
ROC-AUC: 0.8488


In [29]:
matriz_lr = confusion_matrix(y_test, predicciones_lr)
matriz_lr

array([[754, 281],
       [ 81, 293]])

#### Resultado: Regresión Logística en test

Recall: 0.7834 | Precision: 0.5105 | F1: 0.6181 | ROC-AUC: 0.8488

|                     | Predijo: No se va | Predijo: Se va |
|---------------------|--------------------|-----------------|
| Realidad: No se fue | VN = 754           | FP = 281        |
| Realidad: Se fue    | FN = 81            | VP = 293        |

De los 374 clientes que realmente se fueron, el modelo detectó 293 (78.3%), dejando
escapar solo 81. A cambio, genera muchos más falsos positivos que Gradient Boosting
(281 vs 107) — marca como riesgo a bastante más gente que en realidad no se iba a ir.
Consistente con los valores de cross-validation.

### Comparación final: los dos modelos, lado a lado

|                     | Gradient Boosting  | Regresión Logística |
|---------------------|--------------------|-----------------|
|Recall |      	52,4%      |    	78,3%        |
|  Precision  |      	64,7%       |       51,1%        |
|F1|57,9%|	61,8%|
|ROC-AUC|85,3%|84,9%|
|Clientes que se van, detectados|196 de 374|	293 de 374|
|Clientes que se van, NO detectados (FN)|178|81|
|Falsos positivos (campaña gastada de más)|107|281|

### Gráfico: comparación final en test

Se visualizan las métricas finales de ambos modelos en test, y por separado los
Falsos Negativos y Falsos Positivos en cantidad de clientes — el trade-off que se
va a traducir en impacto económico en la Etapa 5.

In [30]:
comparacion_test = pd.DataFrame({
    'Gradient Boosting': [recall_gb, precision_gb, f1_gb, roc_auc_gb],
    'Regresión Logística': [recall_lr, precision_lr, f1_lr, roc_auc_lr]
}, index=['recall', 'precision', 'f1', 'roc_auc'])

comparacion_test_larga = comparacion_test.reset_index().melt(id_vars='index', var_name='Modelo', value_name='Valor')
comparacion_test_larga.columns = ['Métrica', 'Modelo', 'Valor']

fig = px.bar(
    comparacion_test_larga,
    x='Métrica',
    y='Valor',
    color='Modelo',
    barmode='group',
    title='Comparación Final en Test: Gradient Boosting vs Regresión Logística',
    text_auto='.2f'
)
fig.show()

In [31]:
comparacion_errores = pd.DataFrame({
    'Gradient Boosting': [178, 107],
    'Regresión Logística': [81, 281]
}, index=['Falsos Negativos (clientes perdidos)', 'Falsos Positivos (campaña de más)'])

comparacion_errores_larga = comparacion_errores.reset_index().melt(id_vars='index', var_name='Modelo', value_name='Cantidad de clientes')
comparacion_errores_larga.columns = ['Tipo de Error', 'Modelo', 'Cantidad de clientes']

fig2 = px.bar(
    comparacion_errores_larga,
    x='Tipo de Error',
    y='Cantidad de clientes',
    color='Modelo',
    barmode='group',
    title='Trade-off: Clientes Perdidos vs. Campañas Desperdiciadas',
    text_auto=True
)
fig2.show()

#### Resultado: comparación visual final

Se confirma visualmente el trade-off entre ambos modelos: Regresión Logística
detecta muchos más clientes en riesgo real (menor Recall de Falsos Negativos, 81
vs 178), a costa de gastar en más campañas innecesarias (Falsos Positivos, 281 vs
107). Gradient Boosting es la opción inversa: más conservador, menos gasto
desperdiciado, pero deja escapar más clientes reales. La decisión final entre
ambos se resuelve en la Etapa 5, traduciendo este trade-off a impacto económico
concreto con los supuestos de costos definidos en la Etapa 1.

## Cierre de la Etapa 4

Se compararon 3 modelos de clasificación (Regresión Logística, Random Forest,
Gradient Boosting) con Stratified K-Fold Cross-Validation, seleccionando dos
finalistas: Regresión Logística (mejor Recall) y Gradient Boosting (mejor
Precision/ROC-AUC). Se identificó y corrigió un problema de multicolinealidad
entre `Tenure Months` y `Grupo_Antiguedad`, eliminando esta última. Ambos modelos
finalistas fueron evaluados sobre un set de test nunca antes visto, confirmando
consistencia con los resultados de cross-validation (sin señales de overfitting).

**Resultado final en test**:
- Gradient Boosting: Recall 52.4%, Precision 64.7%, F1 57.9%, ROC-AUC 85.3%
- Regresión Logística: Recall 78.3%, Precision 51.1%, F1 61.8%, ROC-AUC 84.9%

El feature importance y los coeficientes confirmaron que Contract, Tenure Months,
Internet Service (especialmente Fiber optic) y Dependents son las variables más
relevantes para predecir churn — consistente con los hallazgos de la exploración
en Excel y los gráficos del notebook 1.

**Próximo paso (Etapa 5)**: traducir la matriz de confusión de ambos modelos a
impacto económico real, usando los supuestos de costos definidos en la Etapa 1,
y determinar cuál modelo (y con qué umbral de decisión) maximiza el ahorro neto
para la empresa.